# Modelado — predicción de supervivencia

Segundo cuaderno del taller. El primero (`02_exploracion.ipynb`) respondía
*"¿qué se asocia con sobrevivir?"*; este responde *"dado un pasajero nuevo,
¿qué probabilidad tiene de sobrevivir?"*.

Lee la capa **Silver** a través de `flows/05_ml_models.py`, nunca los datos
directamente. Para reproducirlo desde cero:

```
make ingest && make silver && make gold
```

y luego *Restart & Run All*.

In [ ]:
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Los modulos con nombre numerico se importan con importlib (seccion 3 de la guia)
ml = importlib.import_module("flows.05_ml_models")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)

X, y = ml.load_dataset()          # lee Silver; X con columnas candidatas, y = Survived
print(X.shape, round(y.mean(), 4))

## Parte 10 — Formulación del problema

Antes de entrenar nada hay que fijar cuatro cosas: la **variable objetivo**, el
**momento de predicción**, la **métrica** y el **baseline** a superar.

El orden importa. Si la métrica se define *después* de entrenar, es fácil
elegir la que mejor le quede al modelo que ya se tiene.

El documento completo está en `reports/problema_ml.md`.

### Paso 39. Qué se predice y cómo se mide

In [ ]:
print(f"Pasajeros:             {len(X)}")
print(f"Variables candidatas:  {X.shape[1]}")
print(f"Tasa de supervivencia: {y.mean():.4f}")
print()
print("Variables que entran al modelo:")
print(f"  numericas:   {ml.NUM_FEATURES}")
print(f"  categoricas: {ml.CAT_FEATURES}")
print(f"  sospechosas: {ml.CABIN_NUM + ml.CABIN_CAT}   <- se evaluan en el Paso 61")
print()
print(f"Metrica principal: {ml.PRIMARY_METRIC}")

### Paso 40. Auditoría de variables y fuga de datos

| Columna | ¿Existe al embarcar? | ¿Riesgo de fuga? | Decisión |
|---|---|---|---|
| `Survived`, `Survivor` | No, es el objetivo | **Fuga directa** | Excluir |
| `PassengerId`, `Ticket`, `Name` | Sí | Identificadores: se memorizan, no generalizan | Excluir (el título entra vía `TTitulo`) |
| `NSex`, `NEmbarked`, `NCabin`, `NTitulo` | — | Duplican a sus categóricas e imponen un orden que no existe | Excluir; se usa *one-hot* |
| `Age` imputada | — | Calculada con el promedio de **todos** los pasajeros | Usar `AgeOriginal` |
| `Person` | Derivada de `Age` y `Sex` | Redundante con ambas | Excluir del modelo base |
| `Cabin`, `TCabin`, `HasCabin` | **Dudoso** | ¿Se registró la cabina *después* del naufragio? | Evaluar con y sin ella (Paso 61) |

**La fuga más sutil es la edad.** La `Age` de Silver se imputó con el promedio
de los 891 pasajeros, incluidos los que terminarán en el conjunto de prueba.
Entrenar con ella le daría al modelo información que no debería tener. Por eso
`load_dataset()` recupera `AgeOriginal`, con sus nulos, y la imputación se hace
dentro del Pipeline, que la calcula solo con datos de entrenamiento.

In [ ]:
# Verificacion: la columna Age que entra al modelo conserva los nulos reales
print(f"Nulos en la Age del modelo: {X['Age'].isna().sum()}   (deben ser 177)")
print()
silver = pd.read_parquet(ml.SILVER / ml.SILVER_NAME)
print("Columnas de Silver que NO entran al modelo:")
for c in sorted(set(silver.columns) - set(X.columns)):
    print(f"  {c}")

**La pregunta de debate del Paso 40.**

Si `HasCabin` mejora la métrica, ¿significa que tener cabina protegía?

Probablemente no. El Paso 5 mostró que la cabina falta en el 18.5 % de primera
clase y en el 97.6 % de tercera: la ausencia del dato refleja **cómo se
compilaron los manifiestos**, no una característica del pasajero. Y cabe la
posibilidad de que se registrara con más detalle para quienes sobrevivieron
para declararla.

Un modelo que explota un artefacto del registro puede tener buenas métricas y
ser inútil para estimar riesgo real. Por eso `TCabin` y `HasCabin` quedan
marcadas como sospechosas y se evalúan por separado en el Paso 61.

### Paso 41. División estratificada

El conjunto de prueba es una evaluación **final** y se abre **una sola vez**,
en la Parte 16. Todas las decisiones intermedias —qué modelo, qué
hiperparámetros, qué umbral, qué variables— se toman con validación cruzada
sobre el entrenamiento.

`stratify=y` mantiene la proporción de sobrevivientes en ambos conjuntos.
`random_state` fija la semilla para que la división sea reproducible.

In [ ]:
X_train, X_test, y_train, y_test = ml.split(X, y)

print(f"Entrenamiento: {len(X_train)} pasajeros, tasa {y_train.mean():.4f}")
print(f"Prueba:        {len(X_test)} pasajeros, tasa {y_test.mean():.4f}")
print(f"Completo:      {len(X)} pasajeros, tasa {y.mean():.4f}")
print()
print(f"Diferencia entre tasas: {abs(y_train.mean() - y_test.mean()):.4f}")

> ⚠️ **A partir de aquí, `X_test` y `y_test` no se tocan** hasta la Parte 16.
> Mirarlos antes invalidaría la evaluación final: cualquier decisión tomada
> viéndolos ajustaría el modelo a ese conjunto concreto, y sus métricas
> dejarían de estimar el rendimiento sobre datos nuevos.

### Paso 42. Baselines: la vara a superar

Un modelo complejo que no supera a una regla simple no aporta valor.

- **Baseline ingenuo:** predecir siempre "no sobrevive".
- **Regla de género:** predecir que sobrevive toda mujer. Es la regla que
  cualquiera deduciría del EDA en cinco minutos.

In [ ]:
ml.baseline_scores(X_train, y_train)

**Lo que muestran estos dos números.**

El baseline ingenuo acierta en torno al 62 % **sin aprender nada**, y su recall
es exactamente 0: no identifica ni un solo superviviente. Es la prueba de que
**la exactitud sola engaña** cuando las clases están desbalanceadas.

La regla de género ronda el 79 % de exactitud. **Esa es la vara real.** Un
modelo de machine learning que quede por debajo no justifica su complejidad.

Por eso la métrica principal es **ROC-AUC** y no la exactitud: no depende del
umbral de decisión y mide la capacidad de ordenar pasajeros por riesgo, que es
lo que una aseguradora necesita.

In [ ]:
# Para tenerlo a mano al comparar modelos en la Parte 13
baselines = ml.baseline_scores(X_train, y_train)
VARA = baselines.set_index("baseline").loc["regla de genero", "accuracy"]
print(f"Vara a superar (exactitud de la regla de genero): {VARA:.4f}")

## Parte 11 — Preprocesamiento reproducible con `Pipeline`

Un `Pipeline` encadena los pasos —imputar, codificar, modelar— en **un solo
objeto** con `fit` y `predict`. Al hacer `fit`, cada paso aprende solo de los
datos que recibe; al predecir, aplica lo aprendido.

Eso garantiza el mismo preprocesamiento en entrenamiento, validación y
producción, sin duplicar código y sin que una estadística del conjunto de
prueba se filtre al de entrenamiento.

### Paso 43. Demostrar la fuga de datos

El experimento: calcular el promedio de edad de las mujeres con **todos** los
datos y con **solo el entrenamiento**, y comparar.

In [ ]:
media_todos = X.loc[X["Sex"] == "female", "Age"].mean()
media_train = X_train.loc[X_train["Sex"] == "female", "Age"].mean()

print(f"Promedio de edad de mujeres, TODOS los datos: {media_todos:.4f}")
print(f"Promedio de edad de mujeres, solo TRAIN:      {media_train:.4f}")
print(f"Diferencia:                                   {abs(media_todos - media_train):.4f}")
print()
print(f"Mujeres en total: {(X['Sex'] == 'female').sum()}")
print(f"Mujeres en train: {(X_train['Sex'] == 'female').sum()}")
print(f"Mujeres en test:  {(X_test['Sex'] == 'female').sum()}")

**Por qué la diferencia es pequeña aquí, y por qué sigue siendo un error.**

La diferencia es de centésimas porque hay cientos de mujeres y el 80 % de
ellas ya está en el entrenamiento: añadir el 20 % restante casi no mueve el
promedio.

Pero el tamaño del error no es lo que lo hace incorrecto. Al usar el promedio
de todos, **información del conjunto de prueba viaja al entrenamiento**: el
modelo "sabe", indirectamente, algo sobre pasajeros que debería desconocer por
completo. La evaluación final deja de estimar el rendimiento sobre datos
nuevos y pasa a estimarlo sobre datos que el modelo ya vio de refilón.

Y en otros conjuntos de datos la diferencia **sí** es grande: con más
faltantes, con grupos pequeños, o con variables de alta varianza, una
estadística calculada sobre todo puede cambiar mucho el resultado. El método
debe ser correcto independientemente de que el error resulte pequeño en este
caso concreto.

### Paso 44. `GroupAgeImputer`

La imputación del Paso 22, ahora como transformador de scikit-learn. Esa
conversión es exactamente lo que elimina la fuga: `fit()` calcula los
promedios con los datos que recibe —dentro de la validación cruzada, solo los
de entrenamiento de ese pliegue— y `transform()` los aplica.

Si aparece un grupo que `fit` nunca vio, usa el promedio global en vez de
dejar un nulo.

In [ ]:
from sklearn.dummy import DummyClassifier

pipe = ml.build_pipeline(DummyClassifier())
pre = pipe[:-1].fit(X_train, y_train)      # solo el preprocesamiento, ajustado en TRAIN

print("Promedios de edad que aprendio el imputador, por (Sex, Pclass):")
print(pre["age_imputer"].group_means_.round(2).to_string())
print(f"\nPromedio global de respaldo: {pre['age_imputer'].global_mean_:.2f}")

### Paso 45-46. Verificar el preprocesamiento

El preprocesador se ajusta con `X_train` y se aplica a `X_test`. Tres
comprobaciones: que no queden nulos, cuántas columnas salen, y que las
categóricas no lleven un orden implícito.

In [ ]:
Z = pre.transform(X_test)

print(f"Entrada:  {X_test.shape[1]} columnas")
print(f"Salida:   {Z.shape[1]} columnas")
print(f"¿Quedan NaN tras el preprocesamiento? {np.isnan(Z).any()}")
print()
print("Nombres de las columnas generadas:")
for nombre in pre["prep"].get_feature_names_out():
    print(f"  {nombre}")

**Las dos cosas que confirma esta salida.**

**No quedan nulos.** Los 177 faltantes de edad los rellenó el `GroupAgeImputer`
con promedios aprendidos *solo* en entrenamiento. Si hubiera quedado algún NaN,
varios modelos fallarían al entrenar.

**`Sex` salió como `cat__Sex_female` y `cat__Sex_male`**, dos columnas de 0 y 1,
no como un único número. Eso es *one-hot*, y evita el problema que anotamos en
el Paso 24: una codificación numérica le diría al modelo que `Q` (2) es mayor
que `S` (0), un orden que no existe.

`min_frequency=10` agrupa las categorías muy raras. La cubierta `T`, con un
solo pasajero, no genera una columna propia que el modelo podría memorizar.

### La decisión a justificar: ¿agrupar por `Sex` o por `Sex` y `Pclass`?

El Paso 22 imputaba la edad con el promedio por género. El pipeline usa un
grupo más específico: género **y** clase.

La comparación formal con validación cruzada llega en el Paso 48. Por ahora,
cuánto difieren los valores que cada variante asignaría.

In [ ]:
por_sexo = ml.GroupAgeImputer(group_cols=("Sex",)).fit(X_train)
por_sexo_clase = ml.GroupAgeImputer(group_cols=("Sex", "Pclass")).fit(X_train)

print("Agrupando solo por Sex:")
print(por_sexo.group_means_.round(2).to_string())
print("\nAgrupando por Sex y Pclass:")
print(por_sexo_clase.group_means_.round(2).to_string())
print("\nRango de valores imputados:")
print(f"  solo Sex:        {por_sexo.group_means_.min():.2f} a {por_sexo.group_means_.max():.2f}")
print(f"  Sex y Pclass:    {por_sexo_clase.group_means_.min():.2f} a {por_sexo_clase.group_means_.max():.2f}")

La agrupación por `Sex` y `Pclass` produce un rango de valores más amplio,
porque recoge algo que el EDA ya mostraba: los pasajeros de primera clase eran
notablemente mayores que los de tercera (Paso 10). Imputar a una mujer de
tercera clase con el promedio de *todas* las mujeres le asignaría una edad
sesgada hacia arriba.

A cambio, los grupos son más pequeños y cada promedio se estima con menos
datos. El Paso 48 dirá si esa especificidad compensa.